# YOLO ile Nesne Tespiti (Yol İşaretleri)

Bu notebook'da **YOLOv8** ile nesne tespiti yapılacaktır.

**Veri Seti:** [Road Sign Detection - andrewmvd/road-sign-detection](https://www.kaggle.com/datasets/andrewmvd/road-sign-detection) (Kaggle API) veya sentetik veri üretimi (fallback)

**Model:** YOLOv8-nano (Ultralytics)

**Konu başlıkları:**
1. Veri setinin indirilmesi / sentetik veri üretimi
2. YOLO formatına dönüştürme
3. YOLOv8 modeli ile eğitim
4. Değerlendirme ve çıkarım

## 0. Gerekli Kütüphaneler

In [ ]:
!pip install -q ultralytics matplotlib numpy pandas pillow

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import yaml
from PIL import Image, ImageDraw
import warnings
warnings.filterwarnings('ignore')

print('Paketler yüklendi!')

## 1. Veri Setini Hazırlama

Kaggle API ile indirme denenir, başarısız olursa **sentetik yol işareti verisi** üretilir.

In [ ]:
import subprocess

DATASET_DIR = Path('./road_sign_dataset')
TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'val'

kaggle_success = False

# Kaggle ile indirmeyi dene
try:
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d',
         'andrewmvd/road-sign-detection',
         '-p', './kaggle_data', '--unzip'],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        kaggle_success = True
        print('Kaggle indirme başarılı!')
    else:
        raise Exception(result.stderr)
except Exception as e:
    print(f'Kaggle indirme başarısız: {e}')
    print('Sentetik veri üretilecek...')

In [ ]:
CLASS_NAMES = ['speed_limit', 'stop', 'crosswalk', 'traffic_light', 'yield']
NUM_CLASSES = len(CLASS_NAMES)

def create_synthetic_dataset(n_train=200, n_val=50, img_size=640):
    """Renkli geometrik şekillerden oluşan sentetik yol işareti verisi üretir."""
    for split, n_images in [('train', n_train), ('val', n_val)]:
        img_dir = DATASET_DIR / split / 'images'
        lbl_dir = DATASET_DIR / split / 'labels'
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for i in range(n_images):
            # Rastgele arka plan rengi
            bg_color = tuple(np.random.randint(100, 255, 3).tolist())
            img = Image.new('RGB', (img_size, img_size), bg_color)
            draw = ImageDraw.Draw(img)

            # Rastgele 1-3 nesne ekle
            n_objects = np.random.randint(1, 4)
            yolo_lines = []

            for _ in range(n_objects):
                cls_id = np.random.randint(0, NUM_CLASSES)
                w = np.random.randint(40, 150)
                h = np.random.randint(40, 150)
                x_center = np.random.randint(w//2 + 5, img_size - w//2 - 5)
                y_center = np.random.randint(h//2 + 5, img_size - h//2 - 5)

                x1 = x_center - w // 2
                y1 = y_center - h // 2
                x2 = x_center + w // 2
                y2 = y_center + h // 2

                # Şekil çiz (sınıfa göre renk)
                colors = ['red', 'blue', 'green', 'yellow', 'white']
                fill_color = colors[cls_id]
                if cls_id == 0:  # speed_limit - daire
                    draw.ellipse([x1, y1, x2, y2], fill=fill_color, outline='black', width=2)
                elif cls_id == 1:  # stop - sekizgen
                    draw.rectangle([x1, y1, x2, y2], fill=fill_color, outline='black', width=2)
                elif cls_id == 2:  # crosswalk - yatay çizgiler
                    draw.rectangle([x1, y1, x2, y2], fill='white', outline='black', width=2)
                    for ly in range(int(y1)+5, int(y2)-5, 8):
                        draw.line([(x1+3, ly), (x2-3, ly)], fill='black', width=2)
                elif cls_id == 3:  # traffic_light - dikdörtgen
                    draw.rectangle([x1, y1, x2, y2], fill='gray', outline='black', width=2)
                    cy = (y1 + y2) // 2
                    draw.ellipse([x1+5, y1+5, x2-5, cy-5], fill='red')
                    draw.ellipse([x1+5, cy, x2-5, y2-5], fill='green')
                else:  # yield - üçgen
                    points = [(x_center, y1), (x1, y2), (x2, y2)]
                    draw.polygon(points, fill=fill_color, outline='black')

                # YOLO formatı: class cx cy w h (normalized)
                cx_norm = x_center / img_size
                cy_norm = y_center / img_size
                w_norm = w / img_size
                h_norm = h / img_size
                yolo_lines.append(f'{cls_id} {cx_norm:.6f} {cy_norm:.6f} {w_norm:.6f} {h_norm:.6f}')

            # Kaydet
            img.save(img_dir / f'{i:05d}.jpg')
            with open(lbl_dir / f'{i:05d}.txt', 'w') as f:
                f.write('\n'.join(yolo_lines))

    print(f'Sentetik veri oluşturuldu: {n_train} train, {n_val} val')
    return True

if not kaggle_success:
    create_synthetic_dataset()
else:
    # Kaggle verisini YOLO formatına dönüştür
    print('Kaggle verisi işlenecek...')
    # (Bu kısım Kaggle veri yapısına göre özelleştirilmeli)

## 2. Veri Yapısını Kontrol Etme

In [ ]:
# YOLO config dosyası oluştur
config = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'train/images',
    'val': 'val/images',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES
}

config_path = DATASET_DIR / 'data.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('YOLO config:')
print(yaml.dump(config, default_flow_style=False))

# Veri sayısını kontrol et
train_imgs = list((DATASET_DIR / 'train' / 'images').glob('*.jpg'))
val_imgs = list((DATASET_DIR / 'val' / 'images').glob('*.jpg'))
print(f'\nTrain resimleri: {len(train_imgs)}')
print(f'Val resimleri: {len(val_imgs)}')

## 3. Örnek Görselleri Gösterme

In [ ]:
def show_sample(img_path, lbl_path, class_names):
    """Bir resmi ve bounding box'larını gösterir."""
    img = Image.open(img_path)
    w, h = img.size

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)

    colors = ['red', 'blue', 'green', 'yellow', 'magenta']

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f.readlines():
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])

                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                box_w = bw * w
                box_h = bh * h

                rect = patches.Rectangle((x1, y1), box_w, box_h,
                                       linewidth=2, edgecolor=colors[cls_id % len(colors)],
                                       facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, class_names[cls_id],
                       color=colors[cls_id % len(colors)], fontsize=10,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_title(f'{img_path.name}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# İlk 3 örneği göster
for i in range(3):
    img_file = sorted(train_imgs)[i]
    lbl_file = DATASET_DIR / 'train' / 'labels' / (img_file.stem + '.txt')
    show_sample(img_file, lbl_file, CLASS_NAMES)

## 4. YOLOv8 Modeli ile Eğitim

Ultralytics YOLOv8-nano modelini transfer öğrenme ile eğiteceğiz.

In [ ]:
from ultralytics import YOLO

# Önceden eğitilmiş YOLOv8-nano modelini yükle
model = YOLO('yolov8n.pt')

print('YOLOv8-nano modeli yüklendi!')
print(f'Model parametreleri: {sum(p.numel() for p in model.model.parameters()):,}')

In [ ]:
# Modeli eğit
results = model.train(
    data=str(config_path),
    epochs=15,
    imgsz=640,
    batch=8,
    name='road_sign_yolo',
    patience=5,
    save=True,
    plots=True
)

print('\nEğitim tamamlandı!')

## 5. Değerlendirme

In [ ]:
# Validation seti üzerinde değerlendir
best_model = YOLO('runs/detect/road_sign_yolo/weights/best.pt')

metrics = best_model.val(data=str(config_path))

print(f'\n=== DEĞERLENDIRME SONUÇLARI ===')
print(f'mAP50:       {metrics.box.map50:.4f}')
print(f'mAP50-95:    {metrics.box.map:.4f}')
print(f'Precision:   {metrics.box.mp:.4f}')
print(f'Recall:      {metrics.box.mr:.4f}')

## 6. Çıkarım (Inference)

In [ ]:
# Val setindeki resimler üzerinde çıkarım yap
val_samples = sorted(val_imgs)[:5]

for img_path in val_samples:
    results = best_model.predict(source=str(img_path), conf=0.5, verbose=False)

    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)

    colors = ['red', 'blue', 'green', 'yellow', 'magenta']

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=colors[cls_id % len(colors)],
                                   facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 8, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                   color='white', fontsize=9,
                   bbox=dict(boxstyle='round', facecolor=colors[cls_id % len(colors)], alpha=0.8))

    ax.set_title(f'Tahmin: {img_path.name}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Sonuç

| Model | mAP50 | Hız | Parametre |
|-------|-------|-----|-----------|
| YOLOv8-nano | Eğitimle belirlenir | Çok hızlı | ~3.2M |
| YOLOv8-small | Daha yüksek | Hızlı | ~11.2M |
| YOLOv8-medium | En yüksek | Yavaş | ~25.9M |

**Öğrenilen dersler:**
- YOLOv8, transfer öğrenme ile az veriyle bile iyi sonuçlar verir
- NMS (Non-Maximum Suppression) çakışan kutuları temizler
- Confidence threshold, precision/receng dengesini etkiler
- Daha büyük model = daha yüksek doğruluk, daha yavaş çıkarım